In [ ]:
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')

Enter your OpenAI API key: ··········


In [ ]:
!pip install -q openai
from openai import OpenAI

client = OpenAI()  # 환경변수에서 자동으로 읽음

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'hello, pilot ready!' in one short sentence."}],
    max_tokens=20,
)
print(resp.choices[0].message.content)

Hello, pilot ready!


# git clone + 데이터 로드

In [ ]:
!git clone https://github.com/Ontology0/Graduation-Project.git
%cd Graduation-Project

Cloning into 'Graduation-Project'...
remote: Enumerating objects: 450, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 450 (delta 124), reused 265 (delta 105), pack-reused 111 (from 1)
Receiving objects: 100% (450/450), 4.69 MiB | 26.23 MiB/s, done.
Resolving deltas: 100% (169/169), done.
/content/Graduation-Project


In [ ]:
!git checkout feat/rag/pilot-pipeline


Branch 'feat/rag/pilot-pipeline' set up to track remote branch 'feat/rag/pilot-pipeline' from 'origin'.
Switched to a new branch 'feat/rag/pilot-pipeline'


In [ ]:
!pip install -q pyyaml python-dotenv


In [ ]:
!git branch -a
print("---")
!git ls-tree -r origin/feat/data/synthetic-conflicts --name-only | findstr synthetic

  dev
* feat/rag/pilot-pipeline
  remotes/origin/HEAD -> origin/dev
  remotes/origin/dev
  remotes/origin/docs/dev브랜치업데이트
  remotes/origin/docs/git-convention
  remotes/origin/docs/pilot-validation-2025-05-26
  remotes/origin/feat/data/synthetic-conflicts
  remotes/origin/feat/rag/pilot-pipeline
  remotes/origin/fix/scaffold-paths
  remotes/origin/main
  remotes/origin/refactoring
---
/bin/bash: line 1: findstr: command not found


In [ ]:
!git branch -a
print("---")
!git ls-tree -r origin/feat/data/synthetic-conflicts --name-only | grep synthetic

  dev
* feat/rag/pilot-pipeline
  remotes/origin/HEAD -> origin/dev
  remotes/origin/dev
  remotes/origin/docs/dev브랜치업데이트
  remotes/origin/docs/git-convention
  remotes/origin/docs/pilot-validation-2025-05-26
  remotes/origin/feat/data/synthetic-conflicts
  remotes/origin/feat/rag/pilot-pipeline
  remotes/origin/fix/scaffold-paths
  remotes/origin/main
  remotes/origin/refactoring
---
data/synthetic/.gitkeep
data/synthetic_conflicts/README.md
data/synthetic_conflicts/pilot_conflicts.jsonl


In [ ]:
!git checkout origin/feat/data/synthetic-conflicts -- data/synthetic_conflicts/

import os
path = "data/synthetic_conflicts/pilot_conflicts.jsonl"
print("exists:", os.path.exists(path))

exists: True


# 실험 코드

## 프롬프트 로드

In [ ]:
import re

def load_prompt(path):
    text = open(path, encoding="utf-8").read()
    system = re.search(r'## System\s*\n(.*?)\n##', text, re.S).group(1).strip()
    user = re.search(r'## User template\s*\n```\s*\n(.*?)\n```', text, re.S).group(1).strip()
    return system, user

base_sys, base_user = load_prompt("configs/prompts/base_rag.md")
ca_sys, ca_user = load_prompt("configs/prompts/conflict_aware.md")

print("=== BASE SYSTEM ===\n", base_sys[:200])
print("\n=== CONFLICT-AWARE SYSTEM ===\n", ca_sys[:200])

=== BASE SYSTEM ===
 You are a helpful assistant. Answer the user's question using the provided context when it is relevant. If the context does not contain enough information, say what is missing briefly.

=== CONFLICT-AWARE SYSTEM ===
 You are a careful RAG assistant. Retrieved documents may conflict with facts you learned during training. Your job is to resolve such **context–memory conflicts** using evidence, time, conditions, and


## 실험 함수 + 실행

In [ ]:
import json

client = OpenAI()
MODEL = "gpt-4o-mini"

cases = [json.loads(l) for l in open("data/synthetic_conflicts/pilot_conflicts.jsonl", encoding="utf-8")]
print(f"Loaded {len(cases)} cases")

def run_case(case, system, user_template):
    ctx = "\n".join(f"[{d['doc_id']}] {d['text']}" for d in case["documents"])
    user_msg = user_template.format(retrieved_context=ctx, question=case["question"])
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_msg},
        ],
        max_tokens=256,
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip()

def run_arm(arm_name, system, user_template):
    results = []
    for i, c in enumerate(cases):
        ans = run_case(c, system, user_template)
        results.append({
            "case_id": c["id"],
            "question": c["question"],
            "gold_answer": c["gold_answer"],
            "conflict_type": c["conflict_type"],
            "predicted_answer": ans,
            "arm": arm_name,
        })
        print(f"[{i+1}/{len(cases)}] {c['id']} done")
    return results

base_results = run_arm("base_rag", base_sys, base_user)
ca_results = run_arm("conflict_aware", ca_sys, ca_user)
print("\nAll done!")

Loaded 10 cases
[1/10] case_001 done
[2/10] case_002 done
[3/10] case_003 done
[4/10] case_004 done
[5/10] case_005 done
[6/10] case_006 done
[7/10] case_007 done
[8/10] case_008 done
[9/10] case_009 done
[10/10] case_010 done
[1/10] case_001 done
[2/10] case_002 done
[3/10] case_003 done
[4/10] case_004 done
[5/10] case_005 done
[6/10] case_006 done
[7/10] case_007 done
[8/10] case_008 done
[9/10] case_009 done
[10/10] case_010 done

All done!


In [ ]:
gold = {c["id"]: c["gold_answer"] for c in cases}
base_map = {r["case_id"]: r for r in base_results}
ca_map = {r["case_id"]: r for r in ca_results}

for cid in sorted(gold):
    print("=" * 70)
    print(f"{cid} ({base_map[cid]['conflict_type']})")
    print(f"Q: {base_map[cid]['question']}")
    print(f"GOLD: {gold[cid]}")
    print(f"\n[BASE]\n{base_map[cid]['predicted_answer'][:300]}")
    print(f"\n[CONFLICT-AWARE]\n{ca_map[cid]['predicted_answer'][:300]}")
    print()

case_001 (temporal)
Q: What is the official primary color of the Helios Research Consortium logo after the 2024 brand refresh?
GOLD: The official primary logo color is solar amber (#F0A202) as defined in the 2024 brand refresh.

[BASE]
The official primary color of the Helios Research Consortium logo after the 2024 brand refresh is solar amber (#F0A202) as stated in the Helios Research Consortium Brand Refresh Memo (2024-03) [d2].

[CONFLICT-AWARE]
There is a conflict between the retrieved evidence and internal knowledge regarding the official primary color of the Helios Research Consortium logo.

According to the retrieved evidence from the Helios Research Consortium Brand Refresh Memo (2024-03) [d2], the official primary logo color is now so

case_002 (version_update)
Q: What is the rated battery capacity of the Orinex Phone Model Z according to the current product specification?
GOLD: The rated battery capacity is 4800 mAh according to the corrected 2024 specification.

[BASE]
The r

1. gpt-4o-mini는 두 arm 모두 10/10 정답

* 10케이스 전부 두 arm 다 정답(gold answer 수치를 맞춤).
* phi-2가 case_002에서 틀렸던 거랑 대조적.의미: 강한 모델은 conflict 처리를 프롬프트 없이도 잘함. → 이게 바로 upper bound야.

2. 답변의 형태에서 차이가 발생
* Base RAG : 바로 정답 + 간결한 근거, 충돌 인지는 '암묵적'
* Conflict-aware : Conlfict exists 라고 명시 후 추론 진행. 어느 문서가 outdated인지도 설명함

ex)case_008
Base: "1 TB, errata가 2TB 오타를 정정함" (바로 정답)
Conflict-aware: "launch announcement는 2TB, 그러나 errata가 정정함" (충돌 구조를 드러냄)

→ 정답은 같지만, conflict-aware는 추론 과정을 투명하게 보여줌. 이건 citation quality / explainability 측면의 차이.



3. 연구에 도움되는 부분
- gpt-4o-mini (upper bound): 충돌 처리 잘함 → "할 수 있는 일"임이 증명됨
- phi-2 (작은 모델): case_002에서 실패 → "작은 모델은 못함"
- Llama 3.1-8B (본 연구 대상): 이 사이 어딘가 → 여기에 RAG-aware DPO를 하면 gpt 수준으로 끌어올릴 수 있나? 가 바로 너 research question

>> 작은 모델은 Conflict에 약함. 큰 모델은 잘함. 그렇다면 8B 모델에 LoRA+RAG-awareDPO로 그 격차를 메울 수 있을 것인가!



4. 현재 실험의 한계
* 데이터셋이 너무 쉽다. Conflict 신호가 명시적임
→ 본 실험에선 더 어려운 케이스 필요
* 현재 평가는 '눈으로 봐서 맞췄나' 수준임. 자동 메트릭 필요

In [ ]:
import json
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"api_pilot_results_{ts}.jsonl"

with open(out_path, "w", encoding="utf-8") as f:
    for r in base_results + ca_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(base_results)+len(ca_results)} results to {out_path}")

# Drive에 백업 (선택)
from google.colab import drive
drive.mount('/content/drive')
!cp {out_path} /content/drive/MyDrive/
print("Backed up to Drive")

Saved 20 results to api_pilot_results_20260528_033348.jsonl
Mounted at /content/drive
Backed up to Drive
